# MESSIDOR × MAPLES-DR — Data Matching & Integration
**Objective:** จับคู่ภาพต้นฉบับ (MESSIDOR) กับ Segmentation Masks (MAPLES-DR)  
สร้าง Verified Catalog ที่พร้อมใช้กับ PyTorch DataLoader

---
## Pipeline Overview
```
MESSIDOR (raw .tif/.jpg)
    └── index file paths  (Step 1)
MAPLES-DR (train/test masks)
    └── scan mask paths   (Step 1)
           │
           ▼
    ID-level matching     (Step 1)
           │
           ▼
    Physical existence    (Step 2)
    verification
           │
           ▼
    Final verified pivot  (Step 2)
           │
           ▼
    Copy to MAPLES_Matched (Step 3)
```

## 0. Configuration & Imports

In [1]:
import os
import glob
import shutil
import pandas as pd
from tqdm import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR        = r"D:\MesidorDataset"
MESSIDOR_DIR    = os.path.join(BASE_DIR, "MESSIDOR")
MAPLES_DR_DIR   = os.path.join(BASE_DIR, "MAPLES-DR")
COLLECTION_DIR  = os.path.join(BASE_DIR, "MAPLES_Matched")

# Derived output paths (created in Step 3)
OUT_IMAGES = os.path.join(COLLECTION_DIR, "images")
OUT_MASKS  = os.path.join(COLLECTION_DIR, "masks")

SPLITS = ["train", "test"]

---
## Step 1 — Index & ID-Level Matching
สร้าง `Image_ID → file path` mapping สำหรับ MESSIDOR  
แล้ว scan Mask ทุกคลาสจาก MAPLES-DR และจับคู่โดยใช้ Image_ID

In [2]:
# ── 1a. Index MESSIDOR images ─────────────────────────────────────────────────
messidor_files = glob.glob(os.path.join(MESSIDOR_DIR, "**", "*.tif"), recursive=True)
if not messidor_files:
    messidor_files = glob.glob(os.path.join(MESSIDOR_DIR, "**", "*.jpg"), recursive=True)

# Map:  stem (no extension) → full path
messidor_map = {os.path.splitext(os.path.basename(f))[0]: f for f in messidor_files}
print(f"MESSIDOR images indexed : {len(messidor_map):,}")

# ── 1b. Scan MAPLES-DR masks and match to MESSIDOR ────────────────────────────
matches = []
for split in SPLITS:
    split_path = os.path.join(MAPLES_DR_DIR, split)
    if not os.path.isdir(split_path):
        print(f"[WARN] Split folder not found: {split_path}")
        continue

    classes = [d for d in os.listdir(split_path)
               if os.path.isdir(os.path.join(split_path, d))]

    for cls in classes:
        cls_path = os.path.join(split_path, cls)
        for m_path in glob.glob(os.path.join(cls_path, "*.png")):
            mask_id = os.path.splitext(os.path.basename(m_path))[0]
            if mask_id in messidor_map:
                matches.append({
                    "Image_ID":            mask_id,
                    "Original_Image_Path": messidor_map[mask_id],
                    "Mask_Path":           m_path,
                    "Class":               cls,
                    "Split":               split,
                })

df_matches = pd.DataFrame(matches)

print(f"Total matching (image, class) pairs : {len(df_matches):,}")
print(f"Unique Image IDs matched            : {df_matches['Image_ID'].nunique():,}")
print(f"Classes found                       : {sorted(df_matches['Class'].unique())}")
display(df_matches.head())

MESSIDOR images indexed : 1,200
Total matching (image, class) pairs : 2,369
Unique Image IDs matched            : 198
Classes found                       : ['BrightUncertains', 'CottonWoolSpots', 'Drusens', 'Exudates', 'Hemorrhages', 'Macula', 'Microaneurysms', 'Neovascularization', 'OpticCup', 'OpticDisc', 'RedUncertains', 'Vessels']


,Image_ID,Original_Image_Path,Mask_Path,Class,Split
0,20051019_38557_0100_PP,D:\MesidorDataset\MESSIDOR\Base11\20051019_385...,D:\MesidorDataset\MAPLES-DR\train\BrightUncert...,BrightUncertains,train
1,20051020_55346_0100_PP,D:\MesidorDataset\MESSIDOR\Base11\20051020_553...,D:\MesidorDataset\MAPLES-DR\train\BrightUncert...,BrightUncertains,train
2,20051020_55701_0100_PP,D:\MesidorDataset\MESSIDOR\Base11\20051020_557...,D:\MesidorDataset\MAPLES-DR\train\BrightUncert...,BrightUncertains,train
3,20051020_58065_0100_PP,D:\MesidorDataset\MESSIDOR\Base11\20051020_580...,D:\MesidorDataset\MAPLES-DR\train\BrightUncert...,BrightUncertains,train
4,20051020_58214_0100_PP,D:\MesidorDataset\MESSIDOR\Base11\20051020_582...,D:\MesidorDataset\MAPLES-DR\train\BrightUncert...,BrightUncertains,train


---
## Step 2 — Physical Existence Verification
ตรวจสอบว่าไฟล์ทั้ง Image และ Mask มีอยู่จริงบน Disk  
→ รายการที่หาย Flag ออก เพื่อให้ DataLoader ไม่ Crash

In [3]:
verified_rows = []
missing_log   = []

for _, row in tqdm(df_matches.iterrows(), total=len(df_matches), desc="Verifying files"):
    img_ok  = os.path.exists(row["Original_Image_Path"])
    mask_ok = os.path.exists(row["Mask_Path"])

    if img_ok and mask_ok:
        verified_rows.append(row)
    else:
        missing_log.append({
            "Image_ID":       row["Image_ID"],
            "Image_Missing":  not img_ok,
            "Mask_Missing":   not mask_ok,
        })

df_verified = pd.DataFrame(verified_rows)

print("=== Verification Summary ===")
print(f"  Verified pairs   : {len(df_verified):,}")
print(f"  Missing entries  : {len(missing_log)}")

if missing_log:
    print("\n[!] Missing files:")
    display(pd.DataFrame(missing_log))

Verifying files: 100%|██████████| 2369/2369 [00:00<00:00, 4474.58it/s]

=== Verification Summary ===
  Verified pairs   : 2,369
  Missing entries  : 0


In [4]:
# ── Pivot to final catalog (one row per image) ─────────────────────────────────
final_catalog = (
    df_verified
    .pivot_table(
        index   = ["Image_ID", "Original_Image_Path", "Split"],
        columns = "Class",
        values  = "Mask_Path",
        aggfunc = "first",
    )
    .reset_index()
)
final_catalog.columns.name = None   # remove pivot artifact

print(f"Unique verified images : {len(final_catalog):,}")
display(final_catalog.head())

Unique verified images : 198


,Image_ID,Original_Image_Path,Split,BrightUncertains,CottonWoolSpots,Drusens,Exudates,Hemorrhages,Macula,Microaneurysms,Neovascularization,OpticCup,OpticDisc,RedUncertains,Vessels
0,20051019_38557_0100_PP,D:\MesidorDataset\MESSIDOR\Base11\20051019_385...,train,D:\MesidorDataset\MAPLES-DR\train\BrightUncert...,D:\MesidorDataset\MAPLES-DR\train\CottonWoolSp...,D:\MesidorDataset\MAPLES-DR\train\Drusens\2005...,D:\MesidorDataset\MAPLES-DR\train\Exudates\200...,D:\MesidorDataset\MAPLES-DR\train\Hemorrhages\...,D:\MesidorDataset\MAPLES-DR\train\Macula\20051...,D:\MesidorDataset\MAPLES-DR\train\Microaneurys...,D:\MesidorDataset\MAPLES-DR\train\Neovasculari...,D:\MesidorDataset\MAPLES-DR\train\OpticCup\200...,D:\MesidorDataset\MAPLES-DR\train\OpticDisc\20...,D:\MesidorDataset\MAPLES-DR\train\RedUncertain...,D:\MesidorDataset\MAPLES-DR\train\Vessels\2005...
1,20051020_44923_0100_PP,D:\MesidorDataset\MESSIDOR\Base11\20051020_449...,test,D:\MesidorDataset\MAPLES-DR\test\BrightUncerta...,D:\MesidorDataset\MAPLES-DR\test\CottonWoolSpo...,D:\MesidorDataset\MAPLES-DR\test\Drusens\20051...,D:\MesidorDataset\MAPLES-DR\test\Exudates\2005...,D:\MesidorDataset\MAPLES-DR\test\Hemorrhages\2...,D:\MesidorDataset\MAPLES-DR\test\Macula\200510...,D:\MesidorDataset\MAPLES-DR\test\Microaneurysm...,D:\MesidorDataset\MAPLES-DR\test\Neovasculariz...,D:\MesidorDataset\MAPLES-DR\test\OpticCup\2005...,D:\MesidorDataset\MAPLES-DR\test\OpticDisc\200...,D:\MesidorDataset\MAPLES-DR\test\RedUncertains...,D:\MesidorDataset\MAPLES-DR\test\Vessels\20051...
2,20051020_55346_0100_PP,D:\MesidorDataset\MESSIDOR\Base11\20051020_553...,train,D:\MesidorDataset\MAPLES-DR\train\BrightUncert...,D:\MesidorDataset\MAPLES-DR\train\CottonWoolSp...,D:\MesidorDataset\MAPLES-DR\train\Drusens\2005...,D:\MesidorDataset\MAPLES-DR\train\Exudates\200...,D:\MesidorDataset\MAPLES-DR\train\Hemorrhages\...,NaN,D:\MesidorDataset\MAPLES-DR\train\Microaneurys...,D:\MesidorDataset\MAPLES-DR\train\Neovasculari...,D:\MesidorDataset\MAPLES-DR\train\OpticCup\200...,D:\MesidorDataset\MAPLES-DR\train\OpticDisc\20...,D:\MesidorDataset\MAPLES-DR\train\RedUncertain...,D:\MesidorDataset\MAPLES-DR\train\Vessels\2005...
3,20051020_55701_0100_PP,D:\MesidorDataset\MESSIDOR\Base11\20051020_557...,train,D:\MesidorDataset\MAPLES-DR\train\BrightUncert...,D:\MesidorDataset\MAPLES-DR\train\CottonWoolSp...,D:\MesidorDataset\MAPLES-DR\train\Drusens\2005...,D:\MesidorDataset\MAPLES-DR\train\Exudates\200...,D:\MesidorDataset\MAPLES-DR\train\Hemorrhages\...,D:\MesidorDataset\MAPLES-DR\train\Macula\20051...,D:\MesidorDataset\MAPLES-DR\train\Microaneurys...,D:\MesidorDataset\MAPLES-DR\train\Neovasculari...,D:\MesidorDataset\MAPLES-DR\train\OpticCup\200...,D:\MesidorDataset\MAPLES-DR\train\OpticDisc\20...,D:\MesidorDataset\MAPLES-DR\train\RedUncertain...,D:\MesidorDataset\MAPLES-DR\train\Vessels\2005...
4,20051020_57566_0100_PP,D:\MesidorDataset\MESSIDOR\Base11\20051020_575...,test,D:\MesidorDataset\MAPLES-DR\test\BrightUncerta...,D:\MesidorDataset\MAPLES-DR\test\CottonWoolSpo...,D:\MesidorDataset\MAPLES-DR\test\Drusens\20051...,D:\MesidorDataset\MAPLES-DR\test\Exudates\2005...,D:\MesidorDataset\MAPLES-DR\test\Hemorrhages\2...,D:\MesidorDataset\MAPLES-DR\test\Macula\200510...,D:\MesidorDataset\MAPLES-DR\test\Microaneurysm...,D:\MesidorDataset\MAPLES-DR\test\Neovasculariz...,D:\MesidorDataset\MAPLES-DR\test\OpticCup\2005...,D:\MesidorDataset\MAPLES-DR\test\OpticDisc\200...,D:\MesidorDataset\MAPLES-DR\test\RedUncertains...,D:\MesidorDataset\MAPLES-DR\test\Vessels\20051...


---
## Step 3 — Data Integration
คัดลอกภาพต้นฉบับและ Mask ที่ Verified แล้วทั้งหมดไปยัง `MAPLES_Matched/`  
โครงสร้างผลลัพธ์:
```
MAPLES_Matched/
├── images/         ← original fundus images
└── masks/
    ├── OpticDisc/
    ├── Macula/
    ├── Exudates/
    └── ...         ← one subfolder per class
```
> **⚠️ Warning:** `MAPLES_Matched/` จะถูก **ลบและสร้างใหม่** ทุกครั้งที่รัน Cell นี้

In [5]:
# ── (Re)create output directory ────────────────────────────────────────────────
if os.path.exists(COLLECTION_DIR):
    shutil.rmtree(COLLECTION_DIR)
os.makedirs(OUT_IMAGES, exist_ok=True)
print(f"Output directory ready: {COLLECTION_DIR}")

# ── Copy images and masks ──────────────────────────────────────────────────────
for _, row in tqdm(final_catalog.iterrows(), total=len(final_catalog), desc="Integrating"):
    img_id  = row["Image_ID"]
    img_src = row["Original_Image_Path"]

    # Copy original image (keep original extension)
    img_ext = os.path.splitext(img_src)[1]
    shutil.copy2(img_src, os.path.join(OUT_IMAGES, f"{img_id}{img_ext}"))

    # Copy all verified masks for this image
    img_masks = df_verified[df_verified["Image_ID"] == img_id]
    for _, mask_row in img_masks.iterrows():
        cls_dir = os.path.join(OUT_MASKS, mask_row["Class"])
        os.makedirs(cls_dir, exist_ok=True)
        shutil.copy2(mask_row["Mask_Path"], os.path.join(cls_dir, f"{img_id}.png"))

# ── Final report ──────────────────────────────────────────────────────────────
n_images  = len(os.listdir(OUT_IMAGES))
n_classes = len(os.listdir(OUT_MASKS))
print("\n=== Integration Complete ===")
print(f"  Images  : {n_images}")
print(f"  Classes : {n_classes}")
print(f"  Output  : {COLLECTION_DIR}")

Output directory ready: D:\MesidorDataset\MAPLES_Matched


Integrating: 100%|██████████| 198/198 [00:05<00:00, 38.76it/s]


=== Integration Complete ===
  Images  : 198
  Classes : 12
  Output  : D:\MesidorDataset\MAPLES_Matched
